# nano4M — Architecture Improvements: Demo Notebook

This notebook lets you load any trained checkpoint and explore its any-to-any generation capabilities.

**What you can do here:**
- Generate any modality conditioned on any other (RGB → Depth → Normals → Caption, Caption → RGB, …)
- Compare baseline vs best variant side-by-side
- Visualise the effect of different architectural choices on generation quality


## 1. Setup

In [ ]:
import os, math
from pathlib import Path
from PIL import Image

import torch
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yaml, omegaconf
from hydra.utils import instantiate

# Point to the project root (works from notebooks/ subfolder too)
root = Path(globals().get('_dh', ['.'])[0]).parent
os.chdir(root)

%load_ext autoreload
%autoreload 2

os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32     = True
torch.set_grad_enabled(False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")


### 1.1 Cosmos tokenizer

In [ ]:
import getpass
from huggingface_hub import snapshot_download
from cosmos_tokenizer.image_lib import ImageTokenizer

TOKENIZER_DIR = f"/scratch/{getpass.getuser()}/nano4M/tokenizers/Cosmos-0.1-Tokenizer-DI16x16"

if not Path(TOKENIZER_DIR).exists():
    print(f"Downloading Cosmos tokenizer to {TOKENIZER_DIR} ...")
    snapshot_download(repo_id='nvidia/Cosmos-0.1-Tokenizer-DI16x16', local_dir=TOKENIZER_DIR)

image_tokenizer = ImageTokenizer(
    checkpoint_enc=f"{TOKENIZER_DIR}/encoder.jit",
    checkpoint_dec=f"{TOKENIZER_DIR}/decoder.jit",
).to(device)
print("Tokenizer ready.")


### 1.2 Dataset

In [ ]:
from nanofm.data.multimodal.simple_multimodal_dataset import SimpleMultimodalDataset

DATASET_ROOT = "/work/com-304/datasets/clevr_com_304/"
MODALITIES   = ["tok_rgb@256", "tok_depth@256", "tok_normal@256", "scene_desc"]

dataset = SimpleMultimodalDataset(
    root_dir=DATASET_ROOT,
    split="val",
    modalities=MODALITIES,
    sample_from_k_augmentations=1,
    text_tokenizer_path="gpt2",
    text_max_length=256,
    transforms=None,
)
print(f"Val set: {len(dataset)} samples")


## 2. Load a model

Change `CHECKPOINT` and `CONFIG` to load any variant.

In [ ]:
try:
    from safetensors.torch import load_file as load_safetensors
except ImportError:
    load_safetensors = None

def load_model(checkpoint_path: str, config_path: str):
    """Instantiate model from YAML config and load safetensors weights."""
    with open(config_path) as f:
        raw = yaml.safe_load(f)
    omegaconf.OmegaConf.register_new_resolver("eval", eval, replace=True)
    cfg = omegaconf.OmegaConf.to_container(omegaconf.OmegaConf.create(raw), resolve=True)

    model = instantiate(cfg["model_config"]).to(device)
    ckpt  = Path(checkpoint_path)
    if ckpt.suffix == ".safetensors":
        sd = load_safetensors(str(ckpt), device=device)
    else:
        sd = torch.load(str(ckpt), map_location=device)
        sd = sd.get("model", sd)
    model.load_state_dict(sd, strict=True)
    model.eval()
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Loaded: {ckpt.parent.name}/{ckpt.name}  ({n/1e6:.2f}M params)")
    return model

# ── Edit these paths ──────────────────────────────────────────────────────────
SCRATCH = f"/scratch/{getpass.getuser()}/nano4M"

CHECKPOINT = f"{SCRATCH}/swiglu_depth16_v1/checkpoint-final.safetensors"
CONFIG     = "cfgs/nano4M/variants/swiglu_depth16.yaml"

model = load_model(CHECKPOINT, CONFIG)


## 3. Helper functions

In [ ]:
def token_ids_to_image(token_ids, to_pil=False):
    """Decode flat token-id tensor → PIL or float tensor (C,H,W) in [0,1]."""    side = int(math.sqrt(token_ids.numel()))
    tokens = token_ids.reshape(1, side, side).to(device)
    with torch.no_grad():
        img = image_tokenizer.decode(tokens)
    img = (img[0].clamp(-1,1).float().cpu() + 1) / 2
    return TF.to_pil_image(img) if to_pil else img


def construct_input(sample_idx: int, input_modality: str):
    """Build encoder input tensors from a val-set sample."""    tokens = dataset[sample_idx][input_modality]
    n = tokens.shape[0]
    enc_tokens    = tokens.unsqueeze(0).to(device)
    enc_positions = torch.arange(n, device=device).unsqueeze(0)
    enc_modalities = (MODALITIES.index(input_modality)
                      * torch.ones(1, n, device=device, dtype=torch.long))
    return enc_tokens, enc_positions, enc_modalities


def generate(model, x_tok, x_pos, x_mod, target_mod: str,
             num_steps=64, temp=0.7, top_p=0.9):
    """Run ROAR decoding and return predicted tokens + updated inputs."""    with torch.no_grad():
        pred, x_tok, x_pos, x_mod = model.generate_one_modality_roar(
            x_tok, x_pos, x_mod, target_mod=target_mod,
            num_steps=num_steps, temp=temp, top_p=top_p, top_k=0.0,
        )
    return pred, x_tok, x_pos, x_mod


def show_token(tokens, modality: str, ax=None, title=""):
    """Display a single token tensor (image or text)."""    if modality == "scene_desc":
        text = dataset.text_tokenizer.decode(tokens[0].cpu().tolist())
        if ax:
            ax.axis("off")
            ax.text(0.5, 0.5, text, ha="center", va="center",
                    wrap=True, fontsize=9, transform=ax.transAxes)
            ax.set_title(title, fontsize=10)
        else:
            print(f"{title}: {text}")
    else:
        img = token_ids_to_image(tokens[0])
        if ax:
            ax.imshow(img.permute(1,2,0))
            ax.axis("off")
            ax.set_title(title, fontsize=10)
        else:
            display(TF.to_pil_image(img))


## 4. Any-to-any generation chains

### 4.1 Caption → RGB → Depth → Normals

In [ ]:
SAMPLE_IDX = 3   # change to explore different samples

chain = [
    ("scene_desc",    "scene_desc",   64, 0.7, 0.9),
    ("tok_rgb@256",   "RGB",          64, 0.7, 0.9),
    ("tok_depth@256", "Depth",        64, 0.7, 0.9),
    ("tok_normal@256","Normals",       64, 0.7, 0.9),
]

# Ground truth row
gt_row = [dataset[SAMPLE_IDX][m] for m, *_ in chain]

# Generation
x_tok, x_pos, x_mod = construct_input(SAMPLE_IDX, "scene_desc")
gen_row = []
for target_mod, label, steps, temp, top_p in chain:
    if target_mod == "scene_desc":
        gen_row.append((x_tok, "scene_desc"))   # use GT caption as seed
    else:
        pred, x_tok, x_pos, x_mod = generate(model, x_tok, x_pos, x_mod,
                                               target_mod, steps, temp, top_p)
        gen_row.append((pred, target_mod))

# Plot
fig, axes = plt.subplots(2, len(chain), figsize=(4*len(chain), 8))
fig.suptitle("Caption → RGB → Depth → Normals", fontsize=13, fontweight="bold")
for col, ((target_mod, label, *_), gt_tokens) in enumerate(zip(chain, gt_row)):
    show_token(gt_tokens.unsqueeze(0), target_mod, ax=axes[0,col],
               title=f"GT — {label}")
for col, (pred_tokens, target_mod) in enumerate(gen_row):
    show_token(pred_tokens, target_mod, ax=axes[1,col],
               title=f"Gen — {chain[col][1]}")
axes[0,0].set_ylabel("Ground truth", fontsize=11)
axes[1,0].set_ylabel("Generated", fontsize=11)
plt.tight_layout()
plt.show()


### 4.2 RGB → Depth → Normals → Caption

In [ ]:
SAMPLE_IDX = 7

x_tok, x_pos, x_mod = construct_input(SAMPLE_IDX, "tok_rgb@256")

targets = [
    ("tok_rgb@256",   "RGB (input)",   None,  None,  None),
    ("tok_depth@256", "Depth",          64,   0.7,   0.9),
    ("tok_normal@256","Normals",        64,   0.7,   0.9),
    ("scene_desc",    "Caption",       128,   0.7,   0.9),
]

fig, axes = plt.subplots(1, len(targets), figsize=(4*len(targets), 4))
fig.suptitle("RGB → Depth → Normals → Caption", fontsize=13, fontweight="bold")

for col, (tmod, label, steps, temp, top_p) in enumerate(targets):
    if steps is None:
        show_token(x_tok, tmod, ax=axes[col], title=label)
    else:
        pred, x_tok, x_pos, x_mod = generate(model, x_tok, x_pos, x_mod,
                                               tmod, steps, temp, top_p)
        show_token(pred, tmod, ax=axes[col], title=label)
plt.tight_layout()
plt.show()


## 5. Baseline vs best variant comparison

Load two models and compare their Caption → RGB generation side by side.

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
BASELINE_CKPT   = f"{SCRATCH}/../outputs/baseline_v1/checkpoint-final.safetensors"
BASELINE_CONFIG = "cfgs/nano4M/multiclevr_d6-6w512.yaml"

BEST_CKPT   = f"{SCRATCH}/swiglu_depth16_v1/checkpoint-final.safetensors"
BEST_CONFIG = "cfgs/nano4M/variants/swiglu_depth16.yaml"
# ──────────────────────────────────────────────────────────────────────────────

model_baseline = load_model(BASELINE_CKPT,   BASELINE_CONFIG)
model_best     = load_model(BEST_CKPT,       BEST_CONFIG)


In [ ]:
N_SAMPLES = 4
fig, axes = plt.subplots(3, N_SAMPLES, figsize=(4*N_SAMPLES, 12))
fig.suptitle("Caption → RGB: Baseline vs Best Variant", fontsize=13, fontweight="bold")
axes[0,0].set_ylabel("Ground Truth", fontsize=11)
axes[1,0].set_ylabel("Baseline (depth6)", fontsize=11)
axes[2,0].set_ylabel("Best variant", fontsize=11)

for col, idx in enumerate(range(N_SAMPLES)):
    gt_tokens = dataset[idx]["tok_rgb@256"]
    show_token(gt_tokens.unsqueeze(0), "tok_rgb@256", ax=axes[0,col], title=f"Sample {idx}")

    for row, mdl in enumerate([model_baseline, model_best], start=1):
        x_tok, x_pos, x_mod = construct_input(idx, "scene_desc")
        pred, *_ = generate(mdl, x_tok, x_pos, x_mod, "tok_rgb@256",
                             num_steps=64, temp=0.7, top_p=0.9)
        show_token(pred, "tok_rgb@256", ax=axes[row,col])

plt.tight_layout()
plt.show()


## 6. Compute-fair evaluation checkpoint

Load an intermediate checkpoint to compare at equal compute budget.

In [ ]:
# Example: depth16 at compute-fair step (3050M tokens ≈ step 22889)
FAIR_CKPT   = f"{SCRATCH}/depth16_v1/checkpoint-22889.safetensors"
FAIR_CONFIG = "cfgs/nano4M/variants/depth16.yaml"

model_cf = load_model(FAIR_CKPT, FAIR_CONFIG)
print("Compute-fair model ready.")


---
*Generated with the nano4M Architecture Improvements project — COM-304, EPFL Spring 2026.*